# **Recomendation Systems**

In [5]:
# Import everything

from surprise import SVD, SVDpp, NMF, Dataset
from surprise.model_selection import GridSearchCV, cross_validate
import pandas as pd
import numpy as np

In [2]:
data = Dataset.load_builtin('ml-100k')
RANDOM_STATE = 42

Dataset ml-100k could not be found. Do you want to download it? [Y/n] Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to C:\Users\user/.surprise_data/ml-100k


In [3]:
baseline_models = {
    "SVD": SVD(random_state=RANDOM_STATE),
    "NMF": NMF(random_state=RANDOM_STATE),
    "SVD++": SVDpp(random_state=RANDOM_STATE),
}

In [6]:
baseline_rows = []
for name, algo in baseline_models.items():
    cv = cross_validate(algo, data, measures=["RMSE", "MAE"], cv=5,
                        n_jobs=-1, verbose=False)
    baseline_rows.append({
        "model": name,
        "RMSE": cv["test_rmse"].mean(),
        "MAE": cv["test_mae"].mean(),
        "fit_time_s": np.mean(cv["fit_time"]),
    })

baseline_df = pd.DataFrame(baseline_rows).set_index("model").round(4)
print("=== Baseline (параметри за замовчуванням) ===")
print(baseline_df)

=== Baseline (параметри за замовчуванням) ===
         RMSE     MAE  fit_time_s
model                            
SVD    0.9372  0.7385      0.5343
NMF    0.9658  0.7594      0.9839
SVD++  0.9197  0.7217     14.7725


In [7]:
grids = {
    "SVD": (SVD, {
        "n_factors": [50, 100, 150],
        "n_epochs": [20, 30],
        "lr_all": [0.002, 0.005, 0.01],
        "reg_all": [0.02, 0.05, 0.1],
    }),
    "NMF": (NMF, {
        "n_factors": [15, 30, 50],
        "n_epochs": [30, 50],
        "biased": [False, True],
        "reg_pu": [0.06, 0.1],
        "reg_qi": [0.06, 0.1],
    }),
    "SVD++": (SVDpp, {
        "n_factors": [20, 40],
        "n_epochs": [20],
        "lr_all": [0.005, 0.01],
        "reg_all": [0.02, 0.05],
    }),
}

In [8]:
best_params = {}
for name, (algo_class, param_grid) in grids.items():
    print(f"\n>>> GridSearchCV: {name}")
    gs = GridSearchCV(algo_class, param_grid, measures=["rmse", "mae"],
                      cv=3, n_jobs=-1, joblib_verbose=0)
    gs.fit(data)
    best_params[name] = gs.best_params["rmse"]
    print("Найкращий RMSE (cv=3):", round(gs.best_score["rmse"], 4))
    print("Параметри:", gs.best_params["rmse"])


>>> GridSearchCV: SVD
Найкращий RMSE (cv=3): 0.9197
Параметри: {'n_factors': 150, 'n_epochs': 30, 'lr_all': 0.01, 'reg_all': 0.1}

>>> GridSearchCV: NMF
Найкращий RMSE (cv=3): 0.9451
Параметри: {'n_factors': 50, 'n_epochs': 50, 'biased': False, 'reg_pu': 0.1, 'reg_qi': 0.1}

>>> GridSearchCV: SVD++
Найкращий RMSE (cv=3): 0.9239
Параметри: {'n_factors': 20, 'n_epochs': 20, 'lr_all': 0.01, 'reg_all': 0.05}


In [10]:
algo_classes = {"SVD": SVD, "NMF": NMF, "SVD++": SVDpp}

final_rows = []
for name, cls in algo_classes.items():
    algo = cls(random_state=RANDOM_STATE, **best_params[name])
    cv = cross_validate(algo, data, measures=["RMSE", "MAE"], cv=5,
                        n_jobs=-1, verbose=False)
    final_rows.append({
        "model": name,
        "RMSE": cv["test_rmse"].mean(),
        "RMSE_std": cv["test_rmse"].std(),
        "MAE": cv["test_mae"].mean(),
        "fit_time_s": np.mean(cv["fit_time"]),
    })

final_df = pd.DataFrame(final_rows).set_index("model").round(4)
print("=== Після підбору параметрів ===")
print(final_df)

best_name = final_df["RMSE"].idxmin()
print(f"\nНайкраща модель за RMSE: {best_name}")


=== Після підбору параметрів ===
         RMSE  RMSE_std     MAE  fit_time_s
model                                      
SVD    0.9119    0.0028  0.7212      1.1616
NMF    0.9372    0.0067  0.7361      2.0304
SVD++  0.9148    0.0056  0.7202     19.0904

Найкраща модель за RMSE: SVD
